<a href="https://colab.research.google.com/github/kzlnyi/ML/blob/main/%22%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%93%D0%BE%D0%BB%D0%BE%D0%B2%D0%B5%D0%BD%D0%BA%D0%BE_%D0%A4%D0%86%D0%A2_3_12_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Головенко Костянтин
**Група:** ФІТ 3-12
**Дата:** 23.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [6]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

print('Користувачів:', ratings_df['user_id'].nunique())
print('Фільмів:', ratings_df['item_id'].nunique())
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', ratings_df['rating'].mean())
print('Пропущені значення:')
print(ratings_df.isnull().sum())

Користувачів: 943
Фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986
Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

У наборі є 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки лежать у діапазоні від 1 до 5, середня оцінка близько 3,5. Найчастіше користувачі ставлять 4, а найрідше 1. Пропущених значень немає, тому дані не потребують додаткового очищення.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [7]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

In [8]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('Розмір train:', len(train_df))
print('Розмір test:', len(testset))

Розмір train: 80000
Розмір test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [9]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD якість залежить від кількості прихованих факторів (n_factors), швидкості навчання (lr_all) і регуляризації (reg_all), яка зменшує перенавчання. Для KNNBasic важливі число сусідів (k), мінімальне число сусідів (min_k), метрика схожості (cosine або pearson) і те, між ким рахується схожість: між користувачами чи між фільмами (user_based). Я залишив стартову сітку з прикладу, бо вона невелика, швидко рахується і містить по два різні значення кожного параметра.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [10]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)

In [11]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(train_data)

In [15]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

print('SVD')
print('Найкращий RMSE:', grid_search_svd.best_score['rmse'])
print('Найкращий MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])
print('Параметри за MAE:', grid_search_svd.best_params['mae'])
print('MAE при параметрах, найкращих за RMSE:',
      grid_search_svd.cv_results['mean_test_mae'][grid_search_svd.best_index['rmse']])

print()
print('KNNBasic')
print('Найкращий RMSE:', grid_search_knn.best_score['rmse'])
print('Найкращий MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])
print('Параметри за MAE:', grid_search_knn.best_params['mae'])
print('MAE при параметрах, найкращих за RMSE:',
      grid_search_knn.cv_results['mean_test_mae'][grid_search_knn.best_index['rmse']])

SVD
Найкращий RMSE: 0.9434456317886474
Найкращий MAE: 0.7439597683611338
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Параметри за MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
MAE при параметрах, найкращих за RMSE: 0.7439597683611338

KNNBasic
Найкращий RMSE: 1.0280203510630506
Найкращий MAE: 0.815517605415217
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Параметри за MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
MAE при параметрах, найкращих за RMSE: 0.815517605415217


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9434 | 0.7440 | n_factors = 50, lr_all = 0.01, reg_all = 0.05 |
| KNNBasic | 1.0280 | 0.8155 | k = 40, min_k = 1, name = cosine, user_based = True |

За результатами кросвалідації SVD показала кращу якість: CV RMSE = 0.9434 і CV MAE = 0.7440, тоді як у KNNBasic RMSE = 1.0280 і MAE = 0.8155. Тобто SVD помиляється в середньому менше приблизно на 0.08 за RMSE і на 0.07 за MAE, тому я обрав її для рекомендацій. Для SVD найкращими параметрами стали n_factors = 50, lr_all = 0.01 і reg_all = 0.05, тобто менша кількість факторів, більша швидкість навчання та сильніша регуляризація з запропонованих значень. Для KNNBasic найкращими виявились k = 40, min_k = 1, метрика cosine та схожість між користувачами. Параметри за RMSE і за MAE в обох моделях збіглися, тому вибір параметрів однозначний.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [16]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [17]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [18]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

У роботі я порівняв моделі SVD і KNNBasic на даних MovieLens 100K. За кросвалідацією кращою виявилась SVD (CV RMSE = 0.9434 проти 1.0280), тому я використав її для рекомендацій. На тесті SVD отримала RMSE = 0.9192 і MAE = 0.7234, а KNNBasic RMSE = 1.0120 і MAE = 0.7984. Результати на тесті близькі до кросвалідації, тож перенавчання немає. Обмеження: для нових користувачів і фільмів без оцінок модель не може зробити надійний прогноз.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [20]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# 8.1
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'
print('Обрана модель:', best_model_name)

# 8.2 - 8.3
try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f'Користувача {user_id} немає у навчальній вибірці')
else:
    rated_items = {inner_iid for inner_iid, _ in trainset.ur[inner_uid]}

    # 8.4
    rows = []
    for inner_iid in trainset.all_items():
        if inner_iid not in rated_items:
            raw_iid = trainset.to_raw_iid(inner_iid)
            rows.append((raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est))

    top10 = (pd.DataFrame(rows, columns=['item_id', 'predicted_rating'])
             .sort_values('predicted_rating', ascending=False)
             .head(10)
             .reset_index(drop=True))
    display(top10)

Обрана модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** У цій роботі я побудував дві рекомендаційні моделі, SVD і KNNBasic, на даних MovieLens 100K і порівняв їх за метриками RMSE та MAE. За результатами кросвалідації кращою виявилась SVD з CV RMSE = 0.9434, тоді як у KNNBasic цей показник дорівнював 1.0280, тому саме SVD я використав для формування рекомендацій. Найкращі параметри SVD: n_factors = 50, lr_all = 0.01, reg_all = 0.05. Для KNNBasic найкращими стали k = 40, min_k = 1, метрика cosine та схожість між користувачами (user_based = True). На незалежній тестовій вибірці SVD показала RMSE = 0.9192 і MAE = 0.7234, а KNNBasic — RMSE = 1.0120 і MAE = 0.7984, тобто SVD залишилась кращою за обома метриками. Результати на тесті трохи кращі за кросвалідацію (різниця близько 0.02), що пояснюється навчанням на повній навчальній вибірці, тому перенавчання немає. Рекомендації мають обмеження: для нових користувачів і нових фільмів, яких не було в навчальних даних, модель не має історії оцінок і не може зробити точний прогноз. Крім того, моделі враховують лише оцінки, а жанри, рік випуску й опис фільмів ігнорують, тому в майбутньому їх можна доповнити такою інформацією.